# تحلیل تعاملی نتایج استدلال ساختارمند

این Notebook برای GitHub Codespaces و VS Code است. فقط نتایج ذخیره‌شده را تحلیل می‌کند و مدل را دوباره اجرا نمی‌کند.

روش‌ها: Direct، CoT و ToT-inspired · داده: GSM8K · مدل: Qwen2.5-0.5B-Instruct

در صورت نیاز: `pip install -r requirements-dashboard.txt`

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown
import ipywidgets as widgets

ROOT = Path.cwd()
if not (ROOT / 'results').exists() and (ROOT.parent / 'results').exists(): ROOT = ROOT.parent
RESULTS = ROOT / 'results'
summary = pd.read_csv(RESULTS / 'summary.csv')
records = [json.loads(x) for x in (RESULTS / 'experiment_results.jsonl').read_text(encoding='utf-8').splitlines() if x.strip()]
details = pd.DataFrame(records)
config_file = RESULTS / 'run_config.json'
config = json.loads(config_file.read_text(encoding='utf-8')) if config_file.exists() else {}
display(Markdown('## تنظیمات اجرا'))
display(pd.Series(config, name='value').to_frame())
display(summary)

In [ ]:
display(Markdown('## مقایسه معیارهای اصلی'))
summary['accuracy_percent'] = summary['accuracy'] * 100
fig = px.bar(summary, x='method', y='accuracy_percent', color='method', text='accuracy_percent', title='دقت پاسخ نهایی', labels={'method':'روش','accuracy_percent':'دقت (%)'})
fig.update_yaxes(range=[0,100], ticksuffix='%'); fig.show()
fig = px.bar(summary, x='method', y='avg_generated_tokens', color='method', text_auto='.1f', title='میانگین توکن تولیدشده'); fig.show()
fig = px.bar(summary, x='method', y='avg_generation_calls', color='method', text_auto='.1f', title='میانگین فراخوانی مدل'); fig.show()
if 'judge_failures' in summary: px.bar(summary, x='method', y='judge_failures', color='method', title='خطاهای قالب داوری').show()

In [ ]:
display(Markdown('## بررسی تعاملی سؤال‌ها و خروجی مدل'))
question_ids = sorted(details['dataset_index'].unique().tolist())
question_picker = widgets.Dropdown(options=question_ids, description='شناسه سؤال:')
method_picker = widgets.Dropdown(options=['Direct','CoT','ToT-inspired'], value='CoT', description='روش:')
output = widgets.Output()
def show_record(_=None):
    with output:
        output.clear_output()
        row = details[(details.dataset_index == question_picker.value) & (details.method == method_picker.value)].iloc[0]
        display(Markdown('### متن مسئله'))
        print(row.get('question',''))
        print('\nپاسخ صحیح:', row.get('gold_answer'))
        print('پاسخ مدل:', row.get('predicted_answer'))
        print('درست:', bool(row.get('correct')))
        print('توکن:', row.get('generated_tokens'), '| فراخوانی:', row.get('generation_calls'))
        raw = row.get('raw_output','')
        if row.get('method') == 'ToT-inspired':
            try:
                parsed = json.loads(raw)
                for i, candidate in enumerate(parsed.get('candidates',[]),1):
                    display(Markdown(f'#### Candidate {i}')); print(candidate)
                display(Markdown('#### Judge output')); print(parsed.get('judge_output',''))
                display(Markdown('#### Selected candidate')); print(parsed.get('selected_candidate_output',''))
            except json.JSONDecodeError: print(raw)
        else: print('\nخروجی خام مدل:\n', raw)
question_picker.observe(show_record, names='value')
method_picker.observe(show_record, names='value')
display(widgets.HBox([question_picker,method_picker]), output)
show_record()

In [ ]:
display(Markdown('## تحلیل ToT-inspired'))
tot = details[details.method == 'ToT-inspired'].copy()
if not tot.empty:
    valid = int(tot['judge_selection_valid'].fillna(False).astype(bool).sum())
    print(f'داوری با قالب معتبر: {valid} از {len(tot)}')
    print(f'خطای قالب داوری: {len(tot)-valid}')
    display(tot[['dataset_index','gold_answer','predicted_answer','candidate_answers','selected_candidate','judge_selection_valid','correct','generated_tokens']])
print('اعتبار قالب داوری لزوماً به معنی درستی محتوایی انتخاب نیست.')

In [ ]:
display(Markdown('## ذخیره جدول سؤال‌به‌سؤال'))
csv_path = RESULTS / 'question_details.csv'
details.to_csv(csv_path, index=False, encoding='utf-8-sig')
print('فایل ساخته شد:', csv_path)
print('از پنل فایل‌های Codespaces روی فایل راست‌کلیک کنید و Download را انتخاب کنید.')

## ملاحظات علمی

- نمونه فعلی فقط ۱۰ سؤال دارد و برای تعمیم آماری کافی نیست.
- ToT-inspired نسخه کامل الگوریتم Tree of Thoughts نیست.
- تولیدکننده و داور یک مدل هستند؛ احتمال سوگیری خودارزیابی وجود دارد.
- تعداد توکن‌ها شاخص حجم خروجی است، نه اندازه‌گیری مستقیم زمان، انرژی یا هزینه مالی.
- پاسخ‌های غیرقابل استخراج در محاسبه دقت نادرست شمرده شده‌اند.